In [0]:
doctors_df = spark.read.csv(
    "/Volumes/ey_data/default/n_vol/doctors.csv",
    header=True,
    inferSchema=True
)

visits_df = spark.read.csv(
    "/Volumes/ey_data/default/n_vol/visits.csv",
    header=True,
    inferSchema=True
)

In [0]:
hospitals_df = spark.read.json(
    "/Volumes/ey_data/default/n_vol/hospital_config.json"
)

In [0]:
doctors_df.show()



+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:


visits_df.show()


+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|       NULL| Pending|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01

In [0]:
hospital_df.write.mode("overwrite").parquet(
    "/Volumes/ey_data/default/n_vol/hospital_temp"
)

hospital_df = spark.read.parquet(
    "/Volumes/ey_data/default/n_vol/hospital_temp"
)

In [0]:
hospital_df = spark.read \
    .option("multiline", "true") \
    .json("/Volumes/ey_data/default/n_vol/hospital_config.json")
hospital_df.printSchema()
hospital_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+-----------------------------+-----------+----------------+------------------------------------+
|city     |contact                      |hospital_id|hospital_name   |services                            |
+---------+-----------------------------+-----------+----------------+------------------------------------+
|Hyderabad|{apollo@mail.com, 9876500011}|H101       |Apollo Hospital |[Cardiology, Neurology, Dermatology]|
|Hyderabad|{yashoda@mail.com, NULL}     |H102       |Yashoda Hospital|[Cardiology, Orthopedics]           |
|Bangalore|{NULL, 9876500013}           |H103       |Care Hospital   |[Neurology, Pediatrics]             |
+---------+-

In [0]:
visits_df.count()

10

In [0]:
from pyspark.sql.functions import *
doctors_df.filter(
    col("city") == "Hyderabad"
).show()


+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doctors_df.filter(
    col("specialization") == "Cardiology"
).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doctors_df.filter(
    col("experience_years") > 10
).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visits_df.filter(
    col("bill_amount") > 5000
).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
from pyspark.sql.functions import col

visits_df.filter(
    col("payment_") == "Pending"
).show()


+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visits_df.filter(
    col("payment_") == "Paid"
).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
doctors_df.groupBy(
    "specialization"
).agg(
    avg("consultation_f")
).show()

+--------------+-------------------+
|specialization|avg(consultation_f)|
+--------------+-------------------+
|   Orthopedics|             2500.0|
|    Cardiology|             2250.0|
|    Pediatrics|             1200.0|
|   Dermatology|             1050.0|
|     Neurology|             1900.0|
+--------------+-------------------+



In [0]:
doctors_df.groupBy(
    "city"
).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|    1|
|  Chennai|    1|
|    Kochi|    1|
|Hyderabad|    2|
|Bangalore|    1|
|     Pune|    1|
|   Mumbai|    1|
+---------+-----+



In [0]:
doctors_df.groupBy(
    "specialization"
).count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|   Orthopedics|    1|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Dermatology|    2|
|     Neurology|    2|
+--------------+-----+



In [0]:
visits_df.agg(
    sum("bill_amount")
).show()

+----------------+
|sum(bill_amount)|
+----------------+
|           46500|
+----------------+



In [0]:
visits_df.agg(
    avg("bill_amount")
).show()

+-----------------+
| avg(bill_amount)|
+-----------------+
|5166.666666666667|
+-----------------+



In [0]:
visits_df.agg(
    max("bill_amount")
).show()

+----------------+
|max(bill_amount)|
+----------------+
|           12000|
+----------------+



In [0]:
visits_df.agg(
    min("bill_amount")
).show()

+----------------+
|min(bill_amount)|
+----------------+
|            1500|
+----------------+



In [0]:
doctors_df.orderBy(
    desc("consultation_f")
).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visits_df.orderBy(
    desc("bill_amount")
).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1008|  Meera Nair|     D103|2026-01

In [0]:
visits_df.filter(
    col("bill_amount").isNull()
).show()


+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visits_df = visits_df.fillna(
    {"bill_amount":0}
)

In [0]:
visits_df = visits_df.withColumn(
    "tax",
    col("bill_amount") * 0.05
)

In [0]:
visits_df = visits_df.withColumn(
    "final_bill",
    col("bill_amount") + col("tax")
)

In [0]:
doctor_visit_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
)

doctor_visit_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
).show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1010| Nisha Reddy|2026-01-14|Heart Checkup|       5500|    Paid|275.0|    5775.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1008|  Meera Nair|2026-01-13| Skin Allergy|          0| Pending|  0.0|       0.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "right"
).show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "full"
).show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D120|       NULL|          NULL|     NULL|            NULL|          NULL|   V1007| Arjun Verma|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
visits_df.join(
    doctors_df,
    "doctor_id",
    "left_anti"
).show()

+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|     D120|   V1007| Arjun Verma|2026-01-13| Migraine|       4000|    Paid|200.0|    4200.0|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+



In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "left_anti"
).show()

+---------+-----------+--------------+-----+----------------+--------------+
|doctor_id|doctor_name|specialization| city|experience_years|consultation_f|
+---------+-----------+--------------+-----+----------------+--------------+
|     D107| Dr. Farhan|     Neurology| Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|Kochi|               9|          1100|
+---------+-----------+--------------+-----+----------------+--------------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_name"
).agg(
    count("visit_id").alias("visit_count")
).show()

+-----------+-----------+
|doctor_name|visit_count|
+-----------+-----------+
| Dr. Ramesh|          2|
|  Dr. Meera|          1|
| Dr. Suresh|          2|
|  Dr. Anita|          2|
|  Dr. Kiran|          1|
|  Dr. Priya|          1|
+-----------+-----------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Ramesh|  10500|
|  Dr. Meera|   1500|
| Dr. Suresh|  18000|
|  Dr. Anita|   2000|
|  Dr. Kiran|   7000|
|  Dr. Priya|   3500|
+-----------+-------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(1)

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|  18000|
+-----------+-------+
only showing top 1 row


In [0]:
doctor_visit_df.groupBy(
    "specialization"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(1)

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|   Orthopedics|  18000|
+--------------+-------+
only showing top 1 row


In [0]:
doctor_visit_df.groupBy(
    "specialization"
).agg(
    avg("bill_amount").alias("avg_revenue")
).show()

+--------------+-----------------+
|specialization|      avg_revenue|
+--------------+-----------------+
|   Orthopedics|           9000.0|
|    Cardiology|5833.333333333333|
|    Pediatrics|           1500.0|
|   Dermatology|           1000.0|
|     Neurology|           3500.0|
+--------------+-----------------+



In [0]:
doctor_visit_df.groupBy(
    "city"
).agg(
    sum("bill_amount").alias("revenue")
).show()

+---------+-------+
|     city|revenue|
+---------+-------+
|    Delhi|   1500|
|  Chennai|   2000|
|Hyderabad|  17500|
|Bangalore|   3500|
|   Mumbai|  18000|
+---------+-------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name"
).agg(
    count("visit_id").alias("visit_count")
).show()

+---------+-----------+-----------+
|doctor_id|doctor_name|visit_count|
+---------+-----------+-----------+
|     D102|  Dr. Priya|          1|
|     D105|  Dr. Meera|          1|
|     D104| Dr. Suresh|          2|
|     D103|  Dr. Anita|          2|
|     D101| Dr. Ramesh|          2|
|     D106|  Dr. Kiran|          1|
+---------+-----------+-----------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name"
).agg(
    sum("bill_amount").alias("revenue")
).orderBy(
    desc("revenue")
).show(3)

+---------+-----------+-------+
|doctor_id|doctor_name|revenue|
+---------+-----------+-------+
|     D104| Dr. Suresh|  18000|
|     D101| Dr. Ramesh|  10500|
|     D106|  Dr. Kiran|   7000|
+---------+-----------+-------+
only showing top 3 rows


In [0]:
doctor_report = doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    count("visit_id").alias("total_visits"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("avg_revenue")
)

doctor_report.show(truncate=False)

+---------+-----------+--------------+---------+------------+-------------+-----------+
|doctor_id|doctor_name|specialization|city     |total_visits|total_revenue|avg_revenue|
+---------+-----------+--------------+---------+------------+-------------+-----------+
|D101     |Dr. Ramesh |Cardiology    |Hyderabad|2           |10500        |5250.0     |
|D104     |Dr. Suresh |Orthopedics   |Mumbai   |2           |18000        |9000.0     |
|D102     |Dr. Priya  |Neurology     |Bangalore|1           |3500         |3500.0     |
|D106     |Dr. Kiran  |Cardiology    |Hyderabad|1           |7000         |7000.0     |
|D105     |Dr. Meera  |Pediatrics    |Delhi    |1           |1500         |1500.0     |
|D103     |Dr. Anita  |Dermatology   |Chennai  |2           |2000         |1000.0     |
+---------+-----------+--------------+---------+------------+-------------+-----------+



In [0]:
hospital_df = spark.read.option("multiline","true").json(
    "/Volumes/ey_data/default/n_vol/hospital_config.json"
)

hospital_df.show(truncate=False)

+---------+-----------------------------+-----------+----------------+------------------------------------+
|city     |contact                      |hospital_id|hospital_name   |services                            |
+---------+-----------------------------+-----------+----------------+------------------------------------+
|Hyderabad|{apollo@mail.com, 9876500011}|H101       |Apollo Hospital |[Cardiology, Neurology, Dermatology]|
|Hyderabad|{yashoda@mail.com, NULL}     |H102       |Yashoda Hospital|[Cardiology, Orthopedics]           |
|Bangalore|{NULL, 9876500013}           |H103       |Care Hospital   |[Neurology, Pediatrics]             |
+---------+-----------------------------+-----------+----------------+------------------------------------+



In [0]:
hospital_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
hospital_flat_df = hospital_df.withColumn(
    "phone",
    col("contact.phone")
)

hospital_flat_df.select(
    "hospital_name",
    "phone"
).show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
hospital_flat_df = hospital_flat_df.withColumn(
    "email",
    col("contact.email")
)

hospital_flat_df.select(
    "hospital_name",
    "email"
).show()


+----------------+----------------+
|   hospital_name|           email|
+----------------+----------------+
| Apollo Hospital| apollo@mail.com|
|Yashoda Hospital|yashoda@mail.com|
|   Care Hospital|            NULL|
+----------------+----------------+



In [0]:
hospital_flat_df.filter(
    col("phone").isNull()
).show()

+---------+--------------------+-----------+----------------+--------------------+-----+----------------+
|     city|             contact|hospital_id|   hospital_name|            services|phone|           email|
+---------+--------------------+-----------+----------------+--------------------+-----+----------------+
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...| NULL|yashoda@mail.com|
+---------+--------------------+-----------+----------------+--------------------+-----+----------------+



In [0]:
hospital_flat_df.filter(
    col("email").isNull()
).show()

+---------+------------------+-----------+-------------+--------------------+----------+-----+
|     city|           contact|hospital_id|hospital_name|            services|     phone|email|
+---------+------------------+-----------+-------------+--------------------+----------+-----+
|Bangalore|{NULL, 9876500013}|       H103|Care Hospital|[Neurology, Pedia...|9876500013| NULL|
+---------+------------------+-----------+-------------+--------------------+----------+-----+



In [0]:
hospital_flat_df = hospital_flat_df.fillna({
    "phone":"Not Available"
})

In [0]:
hospital_flat_df = hospital_flat_df.fillna({
    "email":"Not Available"
})

In [0]:
hospital_flat_df.select(
    "hospital_name",
    "city"
).show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
hospital_flat_df.select(
    "hospital_name",
    "phone"
).show()

+----------------+-------------+
|   hospital_name|        phone|
+----------------+-------------+
| Apollo Hospital|   9876500011|
|Yashoda Hospital|Not Available|
|   Care Hospital|   9876500013|
+----------------+-------------+



In [0]:
from pyspark.sql.functions import explode

services_df = hospital_flat_df.withColumn(
    "service",
    explode("services")
)

In [0]:
services_df.select(
    "hospital_name",
    "service"
).show()

+----------------+-----------+
|   hospital_name|    service|
+----------------+-----------+
| Apollo Hospital| Cardiology|
| Apollo Hospital|  Neurology|
| Apollo Hospital|Dermatology|
|Yashoda Hospital| Cardiology|
|Yashoda Hospital|Orthopedics|
|   Care Hospital|  Neurology|
|   Care Hospital| Pediatrics|
+----------------+-----------+



In [0]:
services_df.groupBy(
    "service"
).count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|Orthopedics|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Dermatology|    1|
|  Neurology|    2|
+-----------+-----+



In [0]:
services_df.filter(
    col("service")=="Cardiology"
).show()

+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+----------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|           email|   service|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+----------+
|Hyderabad|{apollo@mail.com,...|       H101| Apollo Hospital|[Cardiology, Neur...|   9876500011| apollo@mail.com|Cardiology|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com|Cardiology|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+----------+



In [0]:
services_df.filter(
    col("service")=="Neurology"
).show()

+---------+--------------------+-----------+---------------+--------------------+----------+---------------+---------+
|     city|             contact|hospital_id|  hospital_name|            services|     phone|          email|  service|
+---------+--------------------+-----------+---------------+--------------------+----------+---------------+---------+
|Hyderabad|{apollo@mail.com,...|       H101|Apollo Hospital|[Cardiology, Neur...|9876500011|apollo@mail.com|Neurology|
|Bangalore|  {NULL, 9876500013}|       H103|  Care Hospital|[Neurology, Pedia...|9876500013|  Not Available|Neurology|
+---------+--------------------+-----------+---------------+--------------------+----------+---------------+---------+



In [0]:
services_df.filter(
    col("service")=="Orthopedics"
).show()

+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|           email|    service|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----------+
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com|Orthopedics|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----------+



In [0]:
hospital_flat_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/ey_data/default/n_vol/hospital_parquet")

In [0]:
hospital_flat_df.write \
    .mode("overwrite") \
    .option("header",True) \
    .csv("/Volumes/ey_data/default/n_vol/hospital_csv")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8749367699539689>, line 4
      1 hospital_flat_df.write \
      2     .mode("overwrite") \
      3     .option("header",True) \
----> 4     .csv("/Volumes/ey_data/default/n_vol/hospital_csv")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:831, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    812 self.mode(mode)
    813 self._set_opts(
    814     compression=compression,
    815     sep=sep,
   (...)
    829     lineSep=lineSep,
    830 )
--> 831 self.format("csv").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataF

In [0]:
doctor_revenue_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
).groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)
doctor_revenue_df.show()

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|
+---------+-----------+--------------+---------+-------+



In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [0]:
w = Window.orderBy(desc("revenue"))

doctor_revenue_df.withColumn(
    "rank",
    rank().over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_revenue_df.withColumn(
    "dense_rank",
    dense_rank().over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
doctor_revenue_df.withColumn(
    "row_number",
    row_number().over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|row_number|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
doctor_revenue_df.orderBy(
    desc("revenue")
).show(1)

+---------+-----------+--------------+------+-------+
|doctor_id|doctor_name|specialization|  city|revenue|
+---------+-----------+--------------+------+-------+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|  18000|
+---------+-----------+--------------+------+-------+
only showing top 1 row


In [0]:
doctor_revenue_df.orderBy(
    desc("revenue")
).show(3)

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|
+---------+-----------+--------------+---------+-------+
only showing top 3 rows


In [0]:
w = Window.partitionBy(
    "specialization"
).orderBy(
    desc("revenue")
)

doctor_revenue_df.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_revenue_df.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank")<=2
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   2|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
w = Window.orderBy("revenue")

doctor_revenue_df.withColumn(
    "running_revenue",
    sum("revenue").over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+---------------+
|doctor_id|doctor_name|specialization|     city|revenue|running_revenue|
+---------+-----------+--------------+---------+-------+---------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|           1500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|           3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|           7000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|          14000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|          24500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|          42500|
+---------+-----------+--------------+---------+-------+---------------+



In [0]:
doctor_revenue_df.withColumn(
    "previous_revenue",
    lag("revenue").over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|
+---------+-----------+--------------+---------+-------+----------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|            NULL|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|            2000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|            3500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|            7000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|           10500|
+---------+-----------+--------------+---------+-------+----------------+



In [0]:
doctor_revenue_df.withColumn(
    "next_revenue",
    lead("revenue").over(w)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|
+---------+-----------+--------------+---------+-------+------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|        2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|        3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|        7000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|       10500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|       18000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|        NULL|
+---------+-----------+--------------+---------+-------+------------+



In [0]:
doctor_revenue_df.withColumn(
    "previous_revenue",
    lag("revenue").over(w)
).withColumn(
    "difference",
    col("revenue")-col("previous_revenue")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|previous_revenue|difference|
+---------+-----------+--------------+---------+-------+----------------+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|            NULL|      NULL|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|            1500|       500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|            2000|      1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|            3500|      3500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|            7000|      3500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|           10500|      7500|
+---------+-----------+--------------+---------+-------+----------------+----------+



In [0]:
doctor_revenue_df.withColumn(
    "next_revenue",
    lead("revenue").over(w)
).withColumn(
    "difference",
    col("next_revenue")-col("revenue")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+------------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|next_revenue|difference|
+---------+-----------+--------------+---------+-------+------------+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|        2000|       500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|        3500|      1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|        7000|      3500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|       10500|      3500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|       18000|      7500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|        NULL|      NULL|
+---------+-----------+--------------+---------+-------+------------+----------+



In [0]:
w = Window.partitionBy(
    "city"
).orderBy(
    desc("revenue")
)

doctor_revenue_df.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
w = Window.partitionBy(
    "city"
).orderBy(
    "revenue"
)

doctor_revenue_df.withColumn(
    "rank",
    rank().over(w)
).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctor_leaderboard = doctor_revenue_df.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(desc("revenue"))
    )
)

doctor_leaderboard.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctors_df.createOrReplaceTempView("doctors")
visits_df.createOrReplaceTempView("visits")
services_df.createOrReplaceTempView("hospitals")

In [0]:
spark.sql("""
SELECT *
FROM doctors
""").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
spark.sql("""
SELECT specialization,
       COUNT(*) AS total_doctors
FROM doctors
GROUP BY specialization
""").show()

+--------------+-------------+
|specialization|total_doctors|
+--------------+-------------+
|   Orthopedics|            1|
|    Cardiology|            2|
|    Pediatrics|            1|
|   Dermatology|            2|
|     Neurology|            2|
+--------------+-------------+



In [0]:
spark.sql("""
SELECT city,
       COUNT(*) AS total_doctors
FROM doctors
GROUP BY city
""").show()

+---------+-------------+
|     city|total_doctors|
+---------+-------------+
|    Delhi|            1|
|  Chennai|            1|
|    Kochi|            1|
|Hyderabad|            2|
|Bangalore|            1|
|     Pune|            1|
|   Mumbai|            1|
+---------+-------------+



In [0]:
spark.sql("""
SELECT d.doctor_id,
       d.doctor_name,
       SUM(v.bill_amount) AS revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY d.doctor_id,d.doctor_name
""").show()

+---------+-----------+-------+
|doctor_id|doctor_name|revenue|
+---------+-----------+-------+
|     D102|  Dr. Priya|   3500|
|     D105|  Dr. Meera|   1500|
|     D104| Dr. Suresh|  18000|
|     D103|  Dr. Anita|   2000|
|     D101| Dr. Ramesh|  10500|
|     D106|  Dr. Kiran|   7000|
+---------+-----------+-------+



In [0]:
spark.sql("""
SELECT d.specialization,
       SUM(v.bill_amount) AS revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY d.specialization
""").show()

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|   Orthopedics|  18000|
|    Cardiology|  17500|
|    Pediatrics|   1500|
|   Dermatology|   2000|
|     Neurology|   3500|
+--------------+-------+



In [0]:
spark.sql("""
SELECT d.doctor_name,
       SUM(v.bill_amount) revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY d.doctor_name
ORDER BY revenue DESC
LIMIT 5
""").show()

+-----------+-------+
|doctor_name|revenue|
+-----------+-------+
| Dr. Suresh|  18000|
| Dr. Ramesh|  10500|
|  Dr. Kiran|   7000|
|  Dr. Priya|   3500|
|  Dr. Anita|   2000|
+-----------+-------+



In [0]:
spark.sql("""
SELECT *
FROM visits
WHERE payment_='Pending'
""").show()

+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|          0| Pending|  0.0|       0.0|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+



In [0]:
spark.sql("""
SELECT DISTINCT hospital_name
FROM hospitals
WHERE service='Cardiology'
""").show()

+----------------+
|   hospital_name|
+----------------+
| Apollo Hospital|
|Yashoda Hospital|
+----------------+



In [0]:
spark.sql("""
SELECT *
FROM hospitals
WHERE phone='Not Available'
OR email='Not Available'
""").show()

+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|           email|    service|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----------+
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com| Cardiology|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com|Orthopedics|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|   9876500013|   Not Available|  Neurology|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|   9876500013|   Not Available| Pediatrics|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+-----

In [0]:
spark.sql("""
SELECT
d.doctor_id,
d.doctor_name,
d.specialization,
COUNT(v.visit_id) total_visits,
SUM(v.bill_amount) revenue
FROM doctors d
LEFT JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY
d.doctor_id,
d.doctor_name,
d.specialization
ORDER BY revenue DESC
""").show()

+---------+-----------+--------------+------------+-------+
|doctor_id|doctor_name|specialization|total_visits|revenue|
+---------+-----------+--------------+------------+-------+
|     D104| Dr. Suresh|   Orthopedics|           2|  18000|
|     D101| Dr. Ramesh|    Cardiology|           2|  10500|
|     D106|  Dr. Kiran|    Cardiology|           1|   7000|
|     D102|  Dr. Priya|     Neurology|           1|   3500|
|     D103|  Dr. Anita|   Dermatology|           2|   2000|
|     D105|  Dr. Meera|    Pediatrics|           1|   1500|
|     D108|  Dr. Nisha|   Dermatology|           0|   NULL|
|     D107| Dr. Farhan|     Neurology|           0|   NULL|
+---------+-----------+--------------+------------+-------+



In [0]:
doctors_df
visits_df
hospital_df

DataFrame[city: string, contact: struct<email:string,phone:string>, hospital_id: string, hospital_name: string, services: array<string>]

In [0]:
visits_df = visits_df.fillna({
    "bill_amount":0
})

hospital_flat_df = hospital_flat_df.fillna({
    "phone":"Not Available",
    "email":"Not Available"
})

In [0]:
silver_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "left"
)

In [0]:
silver_df = silver_df.withColumn(
    "tax",
    col("bill_amount")*0.05
).withColumn(
    "final_bill",
    col("bill_amount")+col("tax")
)

In [0]:
doctor_rank_df = doctor_revenue_df.withColumn(
    "rank",
    dense_rank().over(
        Window.orderBy(desc("revenue"))
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
specialization_summary = doctor_revenue_df.groupBy(
    "specialization"
).agg(
    sum("revenue").alias("total_revenue"),
    avg("revenue").alias("avg_revenue")
)

specialization_summary.show()

+--------------+-------------+-----------+
|specialization|total_revenue|avg_revenue|
+--------------+-------------+-----------+
|    Cardiology|        17500|     8750.0|
|   Orthopedics|        18000|    18000.0|
|     Neurology|         3500|     3500.0|
|    Pediatrics|         1500|     1500.0|
|   Dermatology|         2000|     2000.0|
+--------------+-------------+-----------+



In [0]:
silver_df.write \
    .mode("overwrite") \
    .parquet("/Volumes/ey_data/default/n_vol/silver_hospital")

In [0]:
doctor_report = doctor_revenue_df

city_report = doctor_revenue_df.groupBy(
    "city"
).agg(
    sum("revenue").alias("total_revenue")
)

specialization_report = specialization_summary

In [0]:
dashboard_df = doctor_revenue_df.join(
    specialization_summary,
    "specialization",
    "left"
)

dashboard_df.show()

+--------------+---------+-----------+---------+-------+-------------+-----------+
|specialization|doctor_id|doctor_name|     city|revenue|total_revenue|avg_revenue|
+--------------+---------+-----------+---------+-------+-------------+-----------+
|    Cardiology|     D101| Dr. Ramesh|Hyderabad|  10500|        17500|     8750.0|
|   Orthopedics|     D104| Dr. Suresh|   Mumbai|  18000|        18000|    18000.0|
|     Neurology|     D102|  Dr. Priya|Bangalore|   3500|         3500|     3500.0|
|    Cardiology|     D106|  Dr. Kiran|Hyderabad|   7000|        17500|     8750.0|
|    Pediatrics|     D105|  Dr. Meera|    Delhi|   1500|         1500|     1500.0|
|   Dermatology|     D103|  Dr. Anita|  Chennai|   2000|         2000|     2000.0|
+--------------+---------+-----------+---------+-------+-------------+-----------+

